In [12]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader

model_name = "facebook/opt-125m"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

def load_data(path: str, name: str):
    data = load_dataset(path, name)
    
    train_data = [x["text"] for x in data["train"] if x["text"] != ""]
    test_data = [x["text"] for x in data["test"] if x["text"] != ""]
    valid_data = [x["text"] for x in data["validation"] if x["text"] != ""]
    
    print("train length : ", len(train_data))
    print("test length : ", len(test_data))
    print("valid length : ", len(valid_data))     
    return train_data, test_data, valid_data

train_data, test_data, valid_data = load_data("Salesforce/wikitext", "wikitext-2-raw-v1")        

c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


train length :  23767
test length :  2891
valid length :  2461


In [13]:
def collate_fn(data):
    batch = tokenizer(
        data,
        return_tensors = "pt",
        padding = True,
        truncation = True,
        max_length = 128,
    )
    
    batch = {k: v.to(device) for k, v in batch.items()}
    return batch


train_loader = DataLoader(
    train_data,
    batch_size = 128,
    shuffle = True,
    collate_fn = collate_fn
)

In [ ]:
acts = {}
def hook_fn(module, inputs, output):
    acts["input"] = inputs[0].detach().float().cpu()
    if isinstance(output, tuple):
        output = output[0]
    acts["output"] = output.detach().float().cpu()

model.eval()
target_module = model.model.decoder.layers.0.fc2
handle = target_module.register_forward_pre_hook(hook_fn)

for batch in train_loader:
    outputs = model(**batch)
    handle.remove
    


torch.Size([128, 128, 50272])
